# Data Cleaning

## 1. Load

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("..")
if not (ROOT / "data" / "train.csv").exists():
    ROOT = Path.cwd() if (Path.cwd() / "data" / "train.csv").exists() else Path.cwd().parent

DATA_DIR = ROOT / "data"
AUX_DIR = DATA_DIR / "auxiliary"

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
hdb = pd.read_csv(AUX_DIR / "sg-hdb-block.csv")
mrt = pd.read_csv(AUX_DIR / "sg-mrt-stations.csv")
schools = pd.read_csv(AUX_DIR / "sg-schools.csv")
malls = pd.read_csv(AUX_DIR / "sg-shopping-malls.csv")
coe = pd.read_csv(AUX_DIR / "sg-coe-prices.csv")

print("train", train.shape, "test", test.shape)
print({k: v.shape for k, v in {"hdb": hdb, "mrt": mrt, "schools": schools, "malls": malls, "coe": coe}.items()})


train (150000, 11) test (50000, 10)
{'hdb': (9464, 11), 'mrt': (243, 8), 'schools': (337, 14), 'malls': (113, 3), 'coe': (539, 8)}


## 2. Listing tables

### Missing values

In [2]:
print("NaN in train", int(train.isna().sum().sum()))
print("NaN in test:", int(test.isna().sum().sum()))

NaN in train 0
NaN in test: 0


### `FLAT_TYPE`

Train mixes `3 room` and `3-room`; test uses hyphens only. Collapse to one token.

In [3]:
def normalise_flat_type(s):
    return (
        s.astype(str).str.lower()
        .str.replace("-", " ")
        .str.replace(r"\s+", " ")
        .str.strip()
    )

print("Before cleaning: ", sorted((train["FLAT_TYPE"]).unique()))
print("After cleaning: ", sorted(normalise_flat_type(train["FLAT_TYPE"]).unique()))


Before cleaning:  ['1 room', '1-room', '2 room', '2-room', '3 room', '3-room', '4 room', '4-room', '5 room', '5-room', 'executive']
After cleaning:  ['1 room', '2 room', '3 room', '4 room', '5 room', 'executive']


### `STREET`

`punggol road` and `Punggol Road` are the same street. Lowercasing halves the label set. `TOWN`, `BLOCK`, and `FLAT_MODEL` get the same `strip` + `lower`.

In [4]:
def normalise_text(s):
    return s.astype(str).str.strip().str.lower().str.replace(r"\s+", " ")

print("train STREET: %d -> %d" % (train["STREET"].nunique(), normalise_text(train["STREET"]).nunique()))
print("test  STREET: %d -> %d" % (test["STREET"].nunique(), normalise_text(test["STREET"]).nunique()))


train STREET: 1146 -> 576
test  STREET: 1129 -> 573


### Constant columns

`FURNISHED` is always `yes`, `FEE` is always `0`. Both are dropped.

In [5]:
CONSTANT_COLS = ["FURNISHED", "FEE"]
print({col: (train[col].unique().tolist(), test[col].unique().tolist()) for col in CONSTANT_COLS})


{'FURNISHED': (['yes'], ['yes']), 'FEE': ([0.0], [0.0])}


### Duplicates

Exact copies add no information: drop them on train after labels are unified (425 → 641 extras). Test stays at 50,000 rows: `example-submission.csv` needs 50,000 results.

Same other features with **different** rent is a parallel lease, not a copy: 17,900 groups, kept.

In [6]:
def labelled(df):
    out = df.copy()
    out["FLAT_TYPE"] = normalise_flat_type(out["FLAT_TYPE"])
    out["STREET"] = normalise_text(out["STREET"])
    return out

train_l = labelled(train)
print("exact extras: train %d -> %d, test %d" % (
    int(train.duplicated().sum()), int(train_l.duplicated().sum()), int(test.duplicated().sum())
))
key = ["RENT_APPROVAL_DATE", "TOWN", "BLOCK", "STREET", "FLAT_TYPE", "FLAT_MODEL", "FLOOR_AREA_SQM", "LEASE_COMMENCE_DATE"]
print("same key, different rent:", int((train_l.groupby(key)["MONTHLY_RENT"].nunique() > 1).sum()), "groups")


exact extras: train 425 -> 641, test 5508
same key, different rent: 17900 groups


## 3. Auxiliary tables

### HDB blocks

`POSTAL_CODE == "NIL"` (285) and `YEAR_COMPLETED` / `MAX_FLOOR == -1` (11) are sentinels → NaN. `TOWN` and `PLANNING_AREA` use different vocabularies; `TOWN` stays the join key.

In [7]:
def clean_hdb(df):
    out = df.copy()
    for col in ("TOWN", "BLOCK", "STREET", "SUBZONE", "PLANNING_AREA", "REGION"):
        out[col] = normalise_text(out[col])
    postal = out["POSTAL_CODE"].astype(str).str.strip()
    out["POSTAL_CODE"] = postal.where(~postal.isin(["NIL", "nil"]), np.nan)
    out["YEAR_COMPLETED"] = out["YEAR_COMPLETED"].replace(-1, np.nan)
    out["MAX_FLOOR"] = out["MAX_FLOOR"].replace(-1, np.nan)
    return out

print("NIL postal:", int((hdb["POSTAL_CODE"].astype(str) == "NIL").sum()))
print("year -1:", int((hdb["YEAR_COMPLETED"] == -1).sum()))

NIL postal: 285
year -1: 11


### MRT

Interchange stations share a name across codes. Eleven names are both `open` and `planned`. `IS_OPEN` marks stations already in service.

In [8]:
def clean_mrt(df):
    out = df.copy()
    for col in ("CODE", "NAME", "STATUS", "SUBZONE", "PLANNING_AREA", "REGION"):
        out[col] = normalise_text(out[col])
    out["IS_OPEN"] = out["STATUS"] == "open"
    return out

print(mrt["STATUS"].value_counts().to_dict())

{'open': 231, 'planned': 12}


### Schools

- `NATRUE_CODE` → `NATURE_CODE`
- `seng kang` → `sengkang`
- integer postals drop a leading zero (`88256` → `088256`)
- expand `ave` / `lor` / `st 41`; `st` + number only, so `st wilfred` is unchanged
- `government-aided sch` → `government-aided school`
- 4 school pairs share coordinates → `SUSPECT_DUPLICATE_COORDS`
- `REGION` (`north/south/east/west`) is not the HDB/MRT region scheme
- `MRT_STATIONS` is free text: case and whitespace only

In [9]:
def expand_school_street(s):
    out = normalise_text(s)
    out = out.str.replace(r"\bave\b", "avenue")
    out = out.str.replace(r"\brd\b", "road")
    out = out.str.replace(r"\bdr\b", "drive")
    out = out.str.replace(r"\blor\b", "lorong")
    out = out.str.replace(r"\bst(?=\s+\d)", "street")
    return out


def clean_schools(df):
    out = df.copy()
    if "NATRUE_CODE" in out.columns:
        out = out.rename(columns={"NATRUE_CODE": "NATURE_CODE"})
    for col in ("NAME", "PLANNING_AREA", "REGION", "TYPE_CODE", "NATURE_CODE", "SESSION_CODE", "MAINLEVEL_CODE"):
        if col in out.columns:
            out[col] = normalise_text(out[col])
    out["PLANNING_AREA"] = out["PLANNING_AREA"].replace({"seng kang": "sengkang"})
    out["TYPE_CODE"] = out["TYPE_CODE"].replace({"government-aided sch": "government-aided school"})
    out["STREET"] = expand_school_street(out["STREET"])
    out["POSTAL_CODE"] = out["POSTAL_CODE"].astype(str).str.replace(r"\.0$", "", n=1).str.zfill(6)
    if "MRT_STATIONS" in out.columns:
        out["MRT_STATIONS"] = normalise_text(out["MRT_STATIONS"])
    if "BUS_LINES" in out.columns:
        out["BUS_LINES"] = normalise_text(out["BUS_LINES"])
    out["SUSPECT_DUPLICATE_COORDS"] = out.duplicated(["LATITUDE", "LONGITUDE"], keep=False)
    return out

print("5-digit postal:", int((schools["POSTAL_CODE"].astype(str).str.len() != 6).sum()))
print("shared-coord pairs:", int(schools.duplicated(["LATITUDE", "LONGITUDE"]).sum()))


5-digit postal: 4
shared-coord pairs: 4


### Shopping malls

Names are Title Case, e.g. `Bugis Junction`. The other tables are lowercase.

In [10]:
def clean_malls(df):
    out = df.copy()
    out["NAME"] = normalise_text(out["NAME"])
    return out

### COE

`QUOTA_PREMIUM` and `PREVAILING_QUOTA_PREMIUM` look like `$40,609`; `QUOTA` and `BIDS_RECEIVED` look like `1,280` — all stored as strings. `MONTH` is `jan` while listings use `2021-01`. Parse the four amount/count columns to numbers; merge `YEAR` + `MONTH` to `YEAR_MONTH`, then drop them.

In [11]:
MONTH_TO_NUM = {
    "jan": 1, "feb": 2, "mar": 3, "apr": 4,
    "may": 5, "jun": 6, "jul": 7, "aug": 8,
    "sep": 9, "oct": 10, "nov": 11, "dec": 12,
}


def parse_numeric_string(s):
    return pd.to_numeric(s.astype(str).str.replace(r"[$,]", ""), errors="coerce")


def clean_coe(df):
    out = df.copy()
    out["CATEGORY"] = normalise_text(out["CATEGORY"])
    out["YEAR_MONTH"] = out["YEAR"].astype(str) + "-" + normalise_text(out["MONTH"]).map(MONTH_TO_NUM).astype(str).str.zfill(2)
    for col in ("QUOTA_PREMIUM", "PREVAILING_QUOTA_PREMIUM", "QUOTA", "BIDS_RECEIVED"):
        out[col] = parse_numeric_string(out[col])
    return out.drop(columns=["YEAR", "MONTH"])

## 4. Assemble

In [12]:
def clean_listings(df, drop_exact_duplicates=False):
    out = df.copy()
    for col in ("TOWN", "BLOCK", "STREET", "FLAT_MODEL"):
        if col in out.columns:
            out[col] = normalise_text(out[col])
    if "FLAT_TYPE" in out.columns:
        out["FLAT_TYPE"] = normalise_flat_type(out["FLAT_TYPE"])
    out = out.drop(columns=[c for c in CONSTANT_COLS if c in out.columns])
    if drop_exact_duplicates:
        out = out.drop_duplicates(keep="first").reset_index(drop=True)
    return out


def clean_auxiliary(hdb_df, mrt_df, schools_df, malls_df, coe_df):
    mrt_c = clean_mrt(mrt_df)
    return {
        "hdb": clean_hdb(hdb_df),
        "mrt": mrt_c,
        "mrt_open": mrt_c[mrt_c["IS_OPEN"]].copy(),
        "schools": clean_schools(schools_df),
        "malls": clean_malls(malls_df),
        "coe": clean_coe(coe_df),
    }


aux = {"hdb": hdb, "mrt": mrt, "schools": schools, "malls": malls, "coe": coe}
train_clean = clean_listings(train, drop_exact_duplicates=True)
test_clean = clean_listings(test, drop_exact_duplicates=False)
aux_clean = clean_auxiliary(hdb, mrt, schools, malls, coe)

print("train", train.shape, "->", train_clean.shape)
print("test ", test.shape, "->", test_clean.shape)
print("aux before cleaning ", {k: v.shape for k, v in aux.items()})
print("aux after cleaning ", {k: v.shape for k, v in aux_clean.items()})


train (150000, 11) -> (149359, 9)
test  (50000, 10) -> (50000, 8)
aux before cleaning  {'hdb': (9464, 11), 'mrt': (243, 8), 'schools': (337, 14), 'malls': (113, 3), 'coe': (539, 8)}
aux after cleaning  {'hdb': (9464, 11), 'mrt': (243, 9), 'mrt_open': (231, 9), 'schools': (337, 15), 'malls': (113, 3), 'coe': (539, 7)}


## 5. Join Check

HDB and COE join on keys. MRT, schools, and malls have no listing key; they join through HDB coordinates.

| Table | How | Key |
|---|---|---|
| hdb | attribute | `TOWN + BLOCK + STREET` |
| coe | month | `RENT_APPROVAL_DATE` = `YEAR_MONTH` |
| mrt / schools / malls | nearest neighbour | listing lat/lon from HDB |

In [13]:
keys = ["TOWN", "BLOCK", "STREET"]


def key_hit(listings, right, on):
    hit = listings.merge(right[on].drop_duplicates(), on=on, how="left", indicator=True)
    return 100 * (hit["_merge"] == "both").mean()


geo = train_clean.merge(aux_clean["hdb"][keys + ["LATITUDE", "LONGITUDE"]], on=keys, how="left")
print("HDB key    train %.2f%%  test %.2f%%" % (
    key_hit(train_clean, aux_clean["hdb"], keys),
    key_hit(test_clean, aux_clean["hdb"], keys),
))
print("COE month  train %.2f%%  test %.2f%%" % (
    100 * train_clean["RENT_APPROVAL_DATE"].isin(aux_clean["coe"]["YEAR_MONTH"]).mean(),
    100 * test_clean["RENT_APPROVAL_DATE"].isin(aux_clean["coe"]["YEAR_MONTH"]).mean(),
))
print("lat/lon missing: listing %d  mrt_open %d  schools %d  malls %d" % (
    int(geo[["LATITUDE", "LONGITUDE"]].isna().any(axis=1).sum()),
    int(aux_clean["mrt_open"][["LATITUDE", "LONGITUDE"]].isna().any(axis=1).sum()),
    int(aux_clean["schools"][["LATITUDE", "LONGITUDE"]].isna().any(axis=1).sum()),
    int(aux_clean["malls"][["LATITUDE", "LONGITUDE"]].isna().any(axis=1).sum()),
))


HDB key    train 100.00%  test 100.00%
COE month  train 100.00%  test 100.00%
lat/lon missing: listing 0  mrt_open 0  schools 0  malls 0


## 6. Summary

This notebook ends in two functions: `clean_listings` and `clean_auxiliary`. They put every table on the same keys and formats so later joins do not break on case, punctuation, or sentinel values.

| Table | Field | Finding | Action |
|---|---|---|---|
| listings | `FLAT_TYPE` | space vs hyphen | `3 room` |
| listings | `STREET` | mixed case | lowercase |
| listings | `FURNISHED`, `FEE` | constant | dropped |
| listings | rows | 641 exact extras after labels | dropped on train; test stays 50,000 |
| listings | property–month key | 17,900 groups with different rent | kept |
| hdb | `POSTAL_CODE`, `YEAR_COMPLETED`, `MAX_FLOOR` | `NIL` / `-1` | NaN |
| hdb | `TOWN` vs `PLANNING_AREA` | different vocab | `TOWN` unchanged |
| mrt | `STATUS` | 11 names also `planned` | `IS_OPEN` / `mrt_open` |
| schools | `NATRUE_CODE`, `PLANNING_AREA`, postal, street | typo, `seng kang`, leading-zero, abbreviations | rename, align, zfill, expand |
| schools | coordinates | 4 shared pairs | `SUSPECT_DUPLICATE_COORDS` |
| schools | `REGION` | not the HDB/MRT scheme | left separate |
| malls | `NAME` | Title Case | lowercase |
| coe | `QUOTA_PREMIUM`, `PREVAILING_QUOTA_PREMIUM`, `QUOTA`, `BIDS_RECEIVED` | `$` / comma strings | numeric |
| coe | `YEAR`, `MONTH` | split time cols | merged to `YEAR_MONTH`, then dropped |